In [3]:
import nltk
import re
import math
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
with open('requirements-3nfr-60fr.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
    non_blank_lines = [line for line in lines if line.strip()]
    first_three = non_blank_lines[:3]
    remaining_lines = non_blank_lines[3:]
    remaining_text = '\n'.join(remaining_lines)
    matrix_NFR = []
    for i, line in enumerate(first_three, start=1):
        matrix_NFR.append([i, line.strip()])
    matrix_sent_texts = []
    for _, nfr_text in matrix_NFR:
        for s in sent_tokenize(nfr_text):
            matrix_sent_texts.append(s.strip())
    remaining_sentences = sent_tokenize(remaining_text)
    filtered_sentences = [s for s in remaining_sentences if s.strip() not in matrix_sent_texts]
    tokenized_matrix = []
    for i, s in enumerate(filtered_sentences, start=1):
        s_strip = s.strip()
        m = re.match(r'(?i)^FR\s*(\d+)\s*[:\-\)]?\s*(.*)$', s_strip)
        content = s_strip
        if m:
            rest = m.group(2)
            content = rest if rest else ''
        tokens = word_tokenize(content) if content else []
        tokens = [t for t in tokens if t != ':']
        tokenized_matrix.append([i, tokens])
    print('matrix_NFR =', matrix_NFR)
    print('tokenized_matrix (per sentence) =', tokenized_matrix)

    # Build token lists for NFR entries (use the original 3 NFR lines)
    nfr_sent_texts = [text for _, text in matrix_NFR]
    nfr_docs = []
    for s in nfr_sent_texts:
        toks = word_tokenize(s)
        cleaned = []
        for t in toks:
            t_clean = re.sub(r"[^\\w\\s]", "", t).lower().strip()
            if not t_clean:
                continue
            if re.search(r'\\d', t_clean):
                continue
            if t_clean in stop_words:
                continue
            cleaned.append(t_clean)
        nfr_docs.append(cleaned)

    docs = [tokens for _, tokens in tokenized_matrix]
    docs_lc = []
    for doc in docs:
        cleaned = []
        for t in doc:
            t_clean = re.sub(r"[^\\w\\s]", "", t).lower().strip()
            if not t_clean:
                continue
            if re.search(r'\\d', t_clean):
                continue
            if t_clean in stop_words:
                continue
            cleaned.append(t_clean)
        docs_lc.append(cleaned)

    docs_all = nfr_docs + docs_lc
    vocab = sorted(set([t for doc in docs_all for t in doc]))

    # Term Frequency per document (for all docs)
    tfs = []
    for doc in docs_all:
        counts = {}
        for t in doc:
            counts[t] = counts.get(t, 0) + 1
        tfs.append(counts)

    # Document Frequency
    df = {w: sum(1 for doc in docs_all if w in doc) for w in vocab}

    # IDF (smoothed)
    N = len(docs_all)
    idf = {w: math.log((N + 1) / (df[w] + 1)) + 1 for w in vocab}

    # TF-IDF vectors for all docs (dicts keyed by vocab terms)
    tfidf_all = []
    for counts in tfs:
        vec = {w: counts.get(w, 0) * idf[w] for w in vocab}
        tfidf_all.append(vec)

    n = len(nfr_docs)
    nfr_tfidf = tfidf_all[:n]
    sent_tfidf = tfidf_all[n:]

    def cosine_dict(a, b):
        dot = 0.0
        for k, v in a.items():
            bv = b.get(k, 0.0)
            if v and bv:
                dot += v * bv
        norm_a = math.sqrt(sum(v * v for v in a.values()))
        norm_b = math.sqrt(sum(v * v for v in b.values()))
        return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

    comparisons = []
    for i, sent in enumerate(filtered_sentences):
        sims = [round(cosine_dict(sent_tfidf[i], nvec), 4) for nvec in nfr_tfidf]
        comparisons.append((i + 1, sent, sims))

    # Print results
    print('vocab size =', len(vocab))
    print('vocab =', vocab)
    print('\nSimilarity of each sentence to the 3 NFR sentences (scores for NFR1,NFR2,NFR3):')
    for idx, sent, sims in comparisons:
        print(f'Sentence {idx}:', sent)
        print('  similarities =', sims)

matrix_NFR = [[1, 'NFR1 (Operational): The system shall interface with the Choice Parts System. This provides the feed of recycled parts data.'], [2, 'NFR2 (Usability): Users shall feel satisfied using the system. 85% of all users will be satisfied with the system.'], [3, 'NFR3 (Security): Only adjusters can request recycled parts audit reports. No users without an adjuster role shall request recycled parts audits.']]
tokenized_matrix (per sentence) = [[1, ['The', 'user', 'shall', 'search', 'for', 'the', 'preferred', 'repair', 'facility', 'using', 'vehicle', 'location', 'and', 'radius', 'in', 'miles', '.']], [2, ['The', 'vehicle', 'location', 'shall', 'include', 'street', 'address', 'city', 'state', 'and', 'zip-code', '.']], [3, ['The', 'search', 'radius', 'shall', 'be', 'between', '1', 'and', '30', 'miles', '.']], [4, ['The', 'system', 'shall', 'locate', 'the', 'preferred', 'repair', 'facility', 'with', 'the', 'highest', 'ratings', 'for', 'the', 'input', 'criteria', '.']], [5, ['The',

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eliza\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
